# Chấm metric cho B1 từ CSV analysis đã có sẵn

Notebook này **không sinh lại B1**. Nó đọc `b1_chrf_scores.csv` đã có trong analysis, ghép source/reference từ `source.csv`, tạo JSONL trung gian đúng schema, rồi dùng cùng metric runner với hai cheat runtime zero-shot/few-shot.

Kết quả gồm CSV B1 chuẩn hóa và CSV metric scores cho syllable/underthesea. B1 hiện có sẵn điểm chrF trong input; notebook vẫn có thể chấm lại 28 cấu hình metric nếu bật metric nặng.


## File cần đặt trong thư mục gốc runtime

- `b1_chrf_scores.csv`: file B1 có sẵn từ `submit/analysis/`.
- `source.csv`: source/reference tương ứng 300 câu.
- `meta-judge-code.zip`: source chạy metric đi kèm notebook.
- `human_all.jsonl`: tùy chọn; cần nếu muốn tính `r_hum` và MC.

CSV B1 phải có `id`, `level`, `operation`, `prediction_vi`; notebook tự thêm `source_zh`, `reference_vi`, `damage_level`, `prompt_type=rule_based`, `status=ok`.


In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()

# Builder tạo hai notebook cố định nhánh; bản generic mặc định dùng few_shot.
BRANCH_NAME = "rule_based"

B1_CSV_FILE = ROOT / "b1_chrf_scores.csv"
SOURCE_FILE = ROOT / "source.csv"
SYNTHETIC_FILE = ROOT / "b1_rule_based.jsonl"
HUMAN_FILE = ROOT / "human_all.jsonl"  # Đặt None nếu chỉ cần score synthetic.
CODE_ZIP = ROOT / "meta-judge-code.zip"

# Khoảng 1-based, lấy cả hai đầu. Ví dụ hai runtime: 1..150 và 151..300.
SENTENCE_START = 1
SENTENCE_END = None  # None = câu cuối cùng trong file.

OUTPUT_BASE = ROOT / "metric-output" / "rule_based"
CACHE_ROOT = ROOT / "metric-cache"

RUN_HEAVY_METRICS = True
RUN_UNDERTHESEA = True
REQUIRE_SIX_LEVELS = True

# None = tự chọn theo CPU/GPU hiện có. Có thể hạ GPU_BATCH_SIZE nếu GPU quá nhỏ.
CPU_WORKERS = None
GPU_BATCH_SIZE = None
STATUS_INTERVAL_SECONDS = 30

## 1. Chuẩn bị runtime

Chọn GPU trong Runtime trước khi chạy. Cell cài thư viện nhẹ vào runtime hiện
tại và giữ package COMET/BERTScore/BLEURT trong ba thư mục riêng để tránh xung
đột NumPy, Pandas, Protobuf và TensorFlow.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from zipfile import ZipFile

if sys.version_info >= (3, 13):
    raise RuntimeError("Hãy chọn Colab runtime dùng Python 3.12 trở xuống.")

if not CODE_ZIP.is_file():
    raise FileNotFoundError(f"Thiếu source bundle: {CODE_ZIP}")

CODE_DIR = ROOT / "meta-judge"
if not (CODE_DIR / "src" / "vn_meta_judge" / "metrics" / "shard.py").is_file():
    with ZipFile(CODE_ZIP) as archive:
        for entry in archive.infolist():
            target = (ROOT / entry.filename).resolve()
            if not target.is_relative_to(ROOT):
                raise ValueError("Source bundle chứa đường dẫn giải nén không hợp lệ.")
        archive.extractall(ROOT)

required_source = CODE_DIR / "src" / "vn_meta_judge" / "metrics" / "shard.py"
if not required_source.is_file():
    raise FileNotFoundError(
        "meta-judge-code.zip chưa có fast metric runner; dùng bundle đi kèm notebook."
    )

CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["NLTK_DATA"] = str(CACHE_ROOT / "nltk")
os.environ["TORCH_HOME"] = str(CACHE_ROOT / "torch")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade-strategy",
        "only-if-needed",
        "-r",
        str(CODE_DIR / "requirements-notebook.txt"),
    ],
    check=True,
)

family_requirements = {
    "BERTScore": "requirements-metric-bertscore.txt",
    "COMET": "requirements-metric-comet.txt",
    "BLEURT": "requirements-metric-bleurt.txt",
}
FAMILY_PACKAGE_PATHS = {}
for family, filename in family_requirements.items():
    requirement = CODE_DIR / filename
    package_dir = CACHE_ROOT / "metric-packages" / family.lower()
    package_dir.mkdir(parents=True, exist_ok=True)
    fingerprint = hashlib.sha256(
        requirement.read_bytes()
        + f"{sys.version_info.major}.{sys.version_info.minor}".encode()
    ).hexdigest()[:16]
    marker = package_dir / f".ready-{fingerprint}"
    if not marker.is_file():
        print(f"Cài package metric: {family}")
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--upgrade",
                "--no-deps",
                "--target",
                str(package_dir),
                "-r",
                str(requirement),
            ],
            check=True,
        )
        marker.write_text("ready\n", encoding="utf-8")
    FAMILY_PACKAGE_PATHS[family] = package_dir

sys.path.insert(0, str(CODE_DIR / "src"))

prefetch_env = os.environ.copy()
prefetch_env["PYTHONPATH"] = str(CODE_DIR / "src")
subprocess.run(
    [
        sys.executable,
        "-m",
        "vn_meta_judge.notebook_workflow",
        "prefetch",
    ],
    cwd=CODE_DIR,
    env=prefetch_env,
    check=True,
)

import torch

cpu_count = os.cpu_count() or 2
CPU_WORKERS = CPU_WORKERS or min(4, max(1, cpu_count))
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    gpu_memory_gb = gpu.total_memory / 1024**3
    GPU_BATCH_SIZE = GPU_BATCH_SIZE or (32 if gpu_memory_gb >= 30 else 16)
    print(f"GPU: {gpu.name} | VRAM: {gpu_memory_gb:.1f} GB")
elif RUN_HEAVY_METRICS:
    raise RuntimeError("Metric nặng đang bật nhưng runtime chưa có GPU.")
else:
    GPU_BATCH_SIZE = GPU_BATCH_SIZE or 4

print("ROOT:", ROOT)
print("CPU workers:", CPU_WORKERS)
print("GPU batch ban đầu:", GPU_BATCH_SIZE)

## 1.5. Chuẩn hóa B1 từ analysis

Bước này chỉ đọc CSV đã có, không gọi API và không gọi hàm sinh B1.


In [ ]:
import csv
import json
from collections import Counter

if not B1_CSV_FILE.is_file():
    raise FileNotFoundError(f"Thiếu file B1 có sẵn: {B1_CSV_FILE}")
if not SOURCE_FILE.is_file():
    raise FileNotFoundError(f"Thiếu source/reference: {SOURCE_FILE}")

with SOURCE_FILE.open(encoding="utf-8-sig", newline="") as handle:
    source_rows = {str(row["ID_cau_VLSP"]): row for row in csv.DictReader(handle)}

with B1_CSV_FILE.open(encoding="utf-8-sig", newline="") as handle:
    b1_rows = list(csv.DictReader(handle))

required = {"id", "level", "operation", "prediction_vi"}
missing = required - set(b1_rows[0]) if b1_rows else required
if missing:
    raise ValueError(f"CSV B1 thiếu cột: {sorted(missing)}")

normalized = []
for row in b1_rows:
    row_id = str(row["id"]).strip()
    if row_id not in source_rows:
        raise ValueError(f"B1 có id không nằm trong source.csv: {row_id}")
    level = int(row["level"])
    if level not in range(6):
        raise ValueError(f"B1 có level ngoài 0..5: {level}")
    source = source_rows[row_id]
    normalized.append({
        "id": row_id,
        "source_zh": source["Nguon_ZH"].strip(),
        "reference_vi": source["Tham_chieu_VI"].strip(),
        "prediction_vi": row["prediction_vi"].strip(),
        "damage_level": level,
        "prompt_type": "rule_based",
        "model_name": "deterministic-rule-baseline-precomputed",
        "backend": "analysis_csv",
        "temperature": 0.0,
        "operation": row["operation"].strip(),
        "status": "ok",
    })

keys = {(row["id"], row["damage_level"]) for row in normalized}
if len(normalized) != 1800 or len(keys) != 1800:
    raise ValueError(f"B1 cần đúng 1,800 dòng/khóa duy nhất, nhận được {len(normalized)} dòng")
for row_id in source_rows:
    levels = sorted(row["damage_level"] for row in normalized if row["id"] == row_id)
    if levels != list(range(6)):
        raise ValueError(f"B1 thiếu level cho {row_id}: {levels}")

with SYNTHETIC_FILE.open("w", encoding="utf-8") as handle:
    for row in normalized:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

normalized_csv = ROOT / "b1_rule_based_normalized.csv"
with normalized_csv.open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(normalized[0]))
    writer.writeheader()
    writer.writerows(normalized)

print("B1 input:", B1_CSV_FILE)
print("B1 JSONL:", SYNTHETIC_FILE, "rows=", len(normalized))
print("B1 normalized CSV:", normalized_csv)
print("Operations:", Counter(row["operation"] for row in normalized))


## 2. Kiểm tra và ghép input

Cell dừng ngay nếu có dòng lỗi API, thiếu văn bản, trùng `id/damage_level`, sai
nhánh hoặc thiếu mức damage. File human tự loại dòng thiếu điểm hoặc có
`quality_flags`; thứ tự các dòng còn lại vẫn được bảo toàn.

In [ ]:
from vn_meta_judge.metrics.shard import prepare_shard_inputs

if not SYNTHETIC_FILE.is_file():
    raise FileNotFoundError(f"Thiếu synthetic input: {SYNTHETIC_FILE}")

resolved_human = HUMAN_FILE
if resolved_human is not None and not Path(resolved_human).is_file():
    raise FileNotFoundError(f"Thiếu human input: {resolved_human}")

input_hasher = hashlib.sha256()
input_hasher.update(BRANCH_NAME.encode())
input_hasher.update(f"{SENTENCE_START}:{SENTENCE_END}".encode())
input_hasher.update(SYNTHETIC_FILE.read_bytes())
if resolved_human is not None:
    input_hasher.update(Path(resolved_human).read_bytes())
range_label = f"{SENTENCE_START}-{SENTENCE_END or 'last'}"
RUN_ID = f"sentences-{range_label}-{input_hasher.hexdigest()[:12]}"

RUN_DIR = OUTPUT_BASE / RUN_ID
DATA_DIR = RUN_DIR / "data"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
RESULT_DIR = RUN_DIR / "results"
LOG_DIR = RUN_DIR / "logs"
for directory in (DATA_DIR, CHECKPOINT_DIR, RESULT_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

COMBINED_FILE = DATA_DIR / "combined.jsonl"
MANIFEST_FILE = DATA_DIR / "input_manifest.json"
input_manifest = prepare_shard_inputs(
    SYNTHETIC_FILE,
    COMBINED_FILE,
    MANIFEST_FILE,
    branch_name=BRANCH_NAME,
    human_path=resolved_human,
    require_six_levels=REQUIRE_SIX_LEVELS,
    sentence_start=SENTENCE_START,
    sentence_end=SENTENCE_END,
)

print(json.dumps(input_manifest, ensure_ascii=False, indent=2))
print("RUN_DIR:", RUN_DIR)

## 3. Bộ lập lịch metric

Metric nhẹ chạy song song theo family trên CPU. Mười hai cấu hình model-based
chạy tuần tự trên một GPU; mỗi cấu hình tự resume từ checkpoint và tự hạ batch
`32 → 16 → 8 → 4 → 2 → 1` khi gặp lỗi thiếu VRAM.

In [ ]:
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

from vn_meta_judge.config import paper_metric_specs

LIGHT_FAMILIES = ("BLEU", "chrF", "ROUGE", "METEOR")
HEAVY_FAMILIES = ("BERTScore", "COMET", "BLEURT")


def safe_name(value):
    return re.sub(r"[^a-zA-Z0-9._-]+", "-", value).strip("-").lower()


def expected_keys(family, selected_key=None):
    if selected_key:
        return [selected_key]
    return [spec.key for spec in paper_metric_specs() if spec.family == family]


def checkpoint_ready(path, family, selected_key=None):
    if not path.is_file():
        return False
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    keys = expected_keys(family, selected_key)
    return all(
        key in payload.get("scores", {})
        and len(payload["scores"][key]) == input_manifest["combined_rows"]
        for key in keys
    )


def task_path(tokenization, family, selected_key=None):
    if selected_key:
        digest = hashlib.sha256(selected_key.encode()).hexdigest()[:10]
        suffix = f"{safe_name(selected_key)[:70]}-{digest}"
    else:
        suffix = "all"
    return CHECKPOINT_DIR / tokenization / f"{safe_name(family)}-{suffix}.json"


def run_task(tokenization, family, batch_size, selected_key=None):
    output = task_path(tokenization, family, selected_key)
    output.parent.mkdir(parents=True, exist_ok=True)
    if checkpoint_ready(output, family, selected_key):
        print(f"SKIP {tokenization} | {family} | {selected_key or 'all'}")
        return output

    label = f"{tokenization}-{family}-{selected_key or 'all'}"
    log_path = LOG_DIR / f"{safe_name(label)}.log"
    command = [
        sys.executable,
        "-m",
        "vn_meta_judge.metrics.shard",
        "worker",
        "--input",
        str(COMBINED_FILE),
        "--output",
        str(output),
        "--tokenization",
        tokenization,
        "--family",
        family,
        "--batch-size",
        str(batch_size),
    ]
    if selected_key:
        command.extend(["--selected-key", selected_key])

    env = os.environ.copy()
    python_paths = [str(CODE_DIR / "src")]
    if family in FAMILY_PACKAGE_PATHS:
        python_paths.insert(0, str(FAMILY_PACKAGE_PATHS[family]))
    env["PYTHONPATH"] = os.pathsep.join(python_paths)
    env["OMP_NUM_THREADS"] = "1"
    env["TOKENIZERS_PARALLELISM"] = "false"
    if family == "COMET":
        env["USE_TF"] = "0"

    started = time.monotonic()
    print(f"START {label} | batch={batch_size}")
    with log_path.open("a", encoding="utf-8") as log:
        log.write("\n$ " + " ".join(command) + "\n")
        process = subprocess.Popen(
            command,
            cwd=CODE_DIR,
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )
        while process.poll() is None:
            time.sleep(min(5, STATUS_INTERVAL_SECONDS))
            elapsed = time.monotonic() - started
            if int(elapsed) > 0 and int(elapsed) % STATUS_INTERVAL_SECONDS < 5:
                print(f"RUNNING {label} | {elapsed / 60:.1f} phút")

    elapsed = time.monotonic() - started
    if process.returncode != 0 or not checkpoint_ready(output, family, selected_key):
        tail = ""
        if log_path.is_file():
            tail = "\n".join(log_path.read_text(encoding="utf-8").splitlines()[-30:])
        raise RuntimeError(
            f"FAILED {label} | batch={batch_size} | log={log_path}\n{tail}"
        )
    print(f"DONE {label} | {elapsed / 60:.1f} phút")
    return output


def is_oom_error(error):
    message = str(error).lower()
    return any(
        marker in message
        for marker in (
            "out of memory",
            "resourceexhausted",
            "cuda error",
            "cudnn_status_alloc_failed",
        )
    )


def adaptive_batches(initial):
    values = []
    value = max(1, int(initial))
    while value >= 1:
        if value not in values:
            values.append(value)
        if value == 1:
            break
        value = max(1, value // 2)
    return values

## 4. Chấm metric nhẹ

BLEU, chrF, ROUGE và METEOR chạy song song. BLEU/chrF/METEOR dùng adapter trực
tiếp tương đương công thức gốc để tránh overhead `evaluate.compute()` từng câu.

In [ ]:
light_tasks = [("syllable", family) for family in LIGHT_FAMILIES]
if RUN_UNDERTHESEA:
    light_tasks.extend(
        [("underthesea", family) for family in ("BLEU", "chrF")]
    )

LIGHT_CHECKPOINTS = []
light_failures = []
with ThreadPoolExecutor(max_workers=CPU_WORKERS) as pool:
    futures = {
        pool.submit(run_task, tokenization, family, 256): (tokenization, family)
        for tokenization, family in light_tasks
    }
    for future in as_completed(futures):
        task = futures[future]
        try:
            LIGHT_CHECKPOINTS.append(future.result())
        except Exception as exc:
            light_failures.append((task, str(exc)))

if light_failures:
    raise RuntimeError("Metric nhẹ lỗi:\n" + json.dumps(light_failures, ensure_ascii=False, indent=2))

print(f"Metric nhẹ hoàn tất: {len(LIGHT_CHECKPOINTS)} task.")

## 5. Chấm metric model-based

BERTScore, COMET và BLEURT dùng GPU. Không tăng số worker GPU trên cùng runtime:
một task tại một thời điểm cho throughput ổn định và tránh hết VRAM.

In [ ]:
HEAVY_CHECKPOINTS = []
if RUN_HEAVY_METRICS:
    heavy_specs = [
        spec for spec in paper_metric_specs() if spec.family in HEAVY_FAMILIES
    ]
    for index, spec in enumerate(heavy_specs, start=1):
        output = task_path("syllable", spec.family, spec.key)
        if checkpoint_ready(output, spec.family, spec.key):
            print(f"[{index:02d}/{len(heavy_specs):02d}] đã có checkpoint: {spec.key}")
            HEAVY_CHECKPOINTS.append(output)
            continue

        last_error = None
        for batch_size in adaptive_batches(GPU_BATCH_SIZE):
            try:
                output = run_task(
                    "syllable",
                    spec.family,
                    batch_size,
                    selected_key=spec.key,
                )
                HEAVY_CHECKPOINTS.append(output)
                last_error = None
                break
            except Exception as exc:
                last_error = exc
                if not is_oom_error(exc) or batch_size == 1:
                    raise
                print(f"OOM ở batch={batch_size}; thử batch nhỏ hơn.")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        if last_error is not None:
            raise last_error
else:
    print("RUN_HEAVY_METRICS=False: chỉ chấm 16 cấu hình metric nhẹ.")

print(f"Metric nặng hoàn tất: {len(HEAVY_CHECKPOINTS)} task.")

## 6. Tách score, tính correlation và đóng gói

Score được tách theo offset đã lưu trong manifest, không sort lại theo `id` hay
`damage_level`. Vì vậy dòng thứ *i* trong `scored_*.jsonl` luôn ứng với dòng đầu
vào thứ *i*; human bị loại giữ `input_index` gốc để truy vết.

In [ ]:
from vn_meta_judge.evaluation.correlations import (
    compute_synthetic_correlations,
    write_correlation_report,
)
from vn_meta_judge.metrics.shard import finalize_shard_scores

all_checkpoints = LIGHT_CHECKPOINTS + HEAVY_CHECKPOINTS
tokenizations = ["syllable"] + (["underthesea"] if RUN_UNDERTHESEA else [])
summaries = {}
for tokenization in tokenizations:
    selected = [
        path
        for path in all_checkpoints
        if path.parent.name == tokenization
    ]
    enabled_families = set(LIGHT_FAMILIES)
    if tokenization == "syllable" and RUN_HEAVY_METRICS:
        enabled_families.update(HEAVY_FAMILIES)
    if tokenization == "underthesea":
        enabled_families = {"BLEU", "chrF"}
    expected_metric_keys = [
        spec.key
        for spec in paper_metric_specs()
        if spec.family in enabled_families
    ]
    summary = finalize_shard_scores(
        MANIFEST_FILE,
        selected,
        RESULT_DIR,
        tokenization=tokenization,
        expected_metric_keys=expected_metric_keys,
    )
    summaries[tokenization] = summary
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    if summary["missing_metrics"] or summary["errors"]:
        raise RuntimeError(
            f"{tokenization} chưa đủ metric; chạy lại từ cell chấm metric."
        )

for tokenization in tokenizations:
    synthetic_scores = RESULT_DIR / f"scores_{BRANCH_NAME}_{tokenization}.json"
    synthetic_rows = RESULT_DIR / f"scored_{BRANCH_NAME}_{tokenization}.jsonl"
    correlation_path = RESULT_DIR / f"correlation_{BRANCH_NAME}_{tokenization}.json"
    if resolved_human is not None:
        human_scores = RESULT_DIR / f"scores_human_{tokenization}.json"
        human_rows = RESULT_DIR / f"scored_human_{tokenization}.jsonl"
        report = write_correlation_report(
            human_rows,
            human_scores,
            synthetic_rows,
            synthetic_scores,
            correlation_path,
        )
        report["scope"] = input_manifest["sentence_range"]
        report["scope_note"] = (
            "Full synthetic input."
            if input_manifest["sentence_range"]["is_full_input"]
            else "Partial sentence range; do not report this MC as the full-branch result."
        )
        correlation_path.write_text(
            json.dumps(report, ensure_ascii=False, indent=2, allow_nan=True),
            encoding="utf-8",
        )
        print(tokenization, "MC:", json.dumps(report["meta"], ensure_ascii=False))
    else:
        synthetic = compute_synthetic_correlations(synthetic_rows, synthetic_scores)
        correlation_path.write_text(
            json.dumps(
                {
                    "synthetic": synthetic,
                    "note": "Chưa có human input nên chỉ tính r_syn, chưa tính MC.",
                    "scope": input_manifest["sentence_range"],
                },
                ensure_ascii=False,
                indent=2,
                allow_nan=True,
            ),
            encoding="utf-8",
        )

archive_base = ROOT / f"metric-results-{BRANCH_NAME}-{RUN_ID}"
archive_path = Path(
    shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR)
)

print("Hoàn tất.")
print("Kết quả:", RESULT_DIR)
print("File tải về:", archive_path)

## 7. Xuất CSV B1 và metric scores

CSV này là output cuối để đưa vào analysis/report; JSONL và checkpoint vẫn giữ để resume.


In [ ]:
import pandas as pd

for tokenization in tokenizations:
    rows_path = RESULT_DIR / f"scored_{BRANCH_NAME}_{tokenization}.jsonl"
    scores_path = RESULT_DIR / f"scores_{BRANCH_NAME}_{tokenization}.json"
    rows = [json.loads(line) for line in rows_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    scores = json.loads(scores_path.read_text(encoding="utf-8"))["scores"]
    frame = pd.DataFrame(rows)
    for key, values in scores.items():
        frame[f"metric__{key}"] = values
    output_csv = RESULT_DIR / f"b1_metric_scores_{tokenization}.csv"
    frame.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print("B1 metric CSV:", output_csv, "rows=", len(frame), "metrics=", len(scores))


In [ ]:
# Kiểm tra nhanh số dòng và năm metric đầu tiên mà không tải toàn bộ bảng vào pandas.
scored_path = RESULT_DIR / f"scored_{BRANCH_NAME}_syllable.jsonl"
with scored_path.open(encoding="utf-8") as handle:
    first_row = json.loads(next(handle))

print("Dòng đầu vào gốc:", first_row["input_index"])
print("ID:", first_row["id"], "| damage_level:", first_row.get("damage_level"))
print("Năm score đầu:")
for key, value in list(first_row["metric_scores"].items())[:5]:
    print(" ", key, "=", value)